In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


In [2]:
project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "Week02_ML_Practical_Projects").is_dir()
)
data_dir = project_root / "Week02_ML_Practical_Projects" / "data" / "raw"

raw_df = pd.read_csv(data_dir / "credit_default_uci350.csv")
metadata = json.loads((data_dir / "uci350_metadata.json").read_text())

feature_names = {
    variable["name"]: variable["description"] or variable["name"]
    for variable in metadata["data"]["variables"]
    if variable["role"] == "Feature"
}
df = raw_df.rename(columns={**feature_names, "Y": "default_next_month"})

print("Data shape:", df.shape)
display(df.head())


Data shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
X = df.drop(columns=["ID", "default_next_month"])
y = df["default_next_month"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values:", int(df.isna().sum().sum()))

class_counts = y.value_counts().sort_index()
class_summary = pd.DataFrame({
    "count": class_counts,
    "proportion": class_counts / len(y),
})
display(class_summary)


X shape: (30000, 23)
y shape: (30000,)
Missing values: 0


,count,proportion
default_next_month,,
0,23364,0.7788
1,6636,0.2212


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

In [5]:
from sklearn.metrics import accuracy_score

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression accuracy: {accuracy_lr:.2%}")

Logistic Regression accuracy: 80.77%


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, digits=3))

[[4528  145]
 [1009  318]]
              precision    recall  f1-score   support

           0      0.818     0.969     0.887      4673
           1      0.687     0.240     0.355      1327

    accuracy                          0.808      6000
   macro avg      0.752     0.604     0.621      6000
weighted avg      0.789     0.808     0.769      6000



In [7]:
LogisticRegression(max_iter=1000, class_weight="balanced")
lr_balanced_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

lr_balanced_model.fit(X_train, y_train)

y_pred_lr_balanced = lr_balanced_model.predict(X_test)
y_prob_lr_balanced = lr_balanced_model.predict_proba(X_test)[:, 1]

In [8]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

print(f"Accuracy: {accuracy_score(y_test, y_pred_lr_balanced):.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr_balanced):.4f}")

print(confusion_matrix(y_test, y_pred_lr_balanced))
print(classification_report(y_test, y_pred_lr_balanced, digits=4))

Accuracy: 67.97%
ROC-AUC: 0.7081
[[3255 1418]
 [ 504  823]]
              precision    recall  f1-score   support

           0     0.8659    0.6966    0.7721      4673
           1     0.3672    0.6202    0.4613      1327

    accuracy                         0.6797      6000
   macro avg     0.6166    0.6584    0.6167      6000
weighted avg     0.7556    0.6797    0.7033      6000



In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

## 训练集内交叉验证与改进

本轮以**违约类 F1**作为模型及阈值的选择标准，同时报告 Precision、Recall、ROC-AUC 和 Average Precision。

- 保留前面的 80% 训练集 / 20% 测试集划分。在 `X_train` 内做 3 折分层交叉验证。
- 每个训练样本的 out-of-fold（OOF，折外）概率，来自没有用该样本训练的模型。
- 使用 OOF 概率选择阈值和候选模型；所有预处理都在各折训练部分拟合。
- OOF 分数参与了模型及阈值选择，是开发分数，不是嵌套验证给出的无偏最终成绩。
- 原测试集此前已多次查看。本轮不使用它选择参数或阈值，最后只做统一的探索性对照评估。

类别编码保留原始取值（包括 0、-2 等）；不凭数字大小推断类别含义，不删除这些记录。


In [10]:
import numpy as np
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
)
from threadpoolctl import threadpool_limits

category_columns = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
]
numeric_columns = [column for column in X_train if column not in category_columns]
amount_columns = [column for column in numeric_columns if column != "AGE"]


def signed_log1p(values):
    # 保留零和负金额；缓和金额特征的长尾。
    return np.sign(values) * np.log1p(np.abs(values))


onehot_preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), category_columns),
    ("numeric", StandardScaler(), numeric_columns),
])

onehot_log_preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), category_columns),
    ("amounts", make_pipeline(
        FunctionTransformer(signed_log1p, feature_names_out="one-to-one"),
        StandardScaler(),
    ), amount_columns),
    ("age", StandardScaler(), ["AGE"]),
])

candidates = {
    "LR": clone(lr_model),
    "LR_balanced": clone(lr_balanced_model),
    "LR_onehot": make_pipeline(onehot_preprocessor, LogisticRegression(max_iter=2000)),
    "LR_onehot_log": make_pipeline(onehot_log_preprocessor, LogisticRegression(max_iter=2000)),
    "RF": clone(rf_model),
    "RF_leaf10": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=10, max_features=0.7,
        random_state=42, n_jobs=-1,
    ),
    "HGB_leaf15": HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=30, l2_regularization=10,
        categorical_features=["SEX", "EDUCATION", "MARRIAGE"],
        early_stopping=True, random_state=42,
    ),
    "HGB_leaf31": HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=30, l2_regularization=10,
        categorical_features=["SEX", "EDUCATION", "MARRIAGE"],
        early_stopping=True, random_state=42,
    ),
}


### 1. 在训练集内部比较模型、选择阈值

One-hot 允许 Logistic Regression 对不同类别分别学习系数；金额的 signed-log 变换保留正负号。
HGB 是直方图梯度提升树；随机森林通过多棵树投票，而提升树逐步修正已有模型的误差。
下面阈值取自 OOF Precision–Recall 曲线中违约 F1 最大的位置，不查看测试集选择。


In [11]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_rows = []
oof_probabilities = {}

for name, estimator in candidates.items():
    oof_probability = np.zeros(len(y_train))
    fold_aucs = []
    # 限制单个模型的数值计算线程，避免多个层次同时占满 CPU。
    with threadpool_limits(limits=4):
        for fit_indices, valid_indices in cv.split(X_train, y_train):
            fold_model = clone(estimator)
            fold_model.fit(X_train.iloc[fit_indices], y_train.iloc[fit_indices])
            fold_probability = fold_model.predict_proba(X_train.iloc[valid_indices])[:, 1]
            oof_probability[valid_indices] = fold_probability
            fold_aucs.append(roc_auc_score(y_train.iloc[valid_indices], fold_probability))

    precision, recall, thresholds = precision_recall_curve(y_train, oof_probability)
    threshold_f1 = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )
    best_index = int(np.argmax(threshold_f1))
    threshold = float(thresholds[best_index])
    oof_prediction = (oof_probability >= threshold).astype(int)
    oof_probabilities[name] = oof_probability
    cv_rows.append({
        "model": name,
        "cv_auc_mean": np.mean(fold_aucs),
        "cv_auc_std": np.std(fold_aucs, ddof=1),
        "oof_average_precision": average_precision_score(y_train, oof_probability),
        "oof_f1_at_0_5": f1_score(y_train, oof_probability > 0.5),
        "threshold": threshold,
        "oof_precision": precision_score(y_train, oof_prediction, zero_division=0),
        "oof_recall": recall_score(y_train, oof_prediction),
        "oof_f1": f1_score(y_train, oof_prediction),
    })
    print(f"{name}: CV AUC={np.mean(fold_aucs):.4f}, OOF F1={threshold_f1[best_index]:.4f}, threshold={threshold:.4f}")

cv_comparison = pd.DataFrame(cv_rows).sort_values(
    ["oof_f1", "cv_auc_mean"], ascending=False,
).reset_index(drop=True)
display(cv_comparison.round(4))


LR: CV AUC=0.7260, OOF F1=0.5152, threshold=0.2733
LR_balanced: CV AUC=0.7263, OOF F1=0.5129, threshold=0.5823
LR_onehot: CV AUC=0.7685, OOF F1=0.5378, threshold=0.2533


LR_onehot_log: CV AUC=0.7747, OOF F1=0.5395, threshold=0.2631


RF: CV AUC=0.7674, OOF F1=0.5336, threshold=0.3350


RF_leaf10: CV AUC=0.7791, OOF F1=0.5463, threshold=0.2859


HGB_leaf15: CV AUC=0.7832, OOF F1=0.5475, threshold=0.2482


HGB_leaf31: CV AUC=0.7837, OOF F1=0.5475, threshold=0.2771


,model,cv_auc_mean,cv_auc_std,oof_average_precision,oof_f1_at_0_5,threshold,oof_precision,oof_recall,oof_f1
0,HGB_leaf31,0.7837,0.0072,0.5512,0.4756,0.2771,0.5281,0.5685,0.5475
1,HGB_leaf15,0.7832,0.0065,0.5542,0.4765,0.2482,0.5009,0.6037,0.5475
2,RF_leaf10,0.7791,0.0039,0.5543,0.4743,0.2859,0.5231,0.5717,0.5463
3,LR_onehot_log,0.7747,0.0069,0.5475,0.4718,0.2631,0.5248,0.5551,0.5395
4,LR_onehot,0.7685,0.0096,0.5411,0.4713,0.2533,0.5269,0.5491,0.5378
5,RF,0.7674,0.0047,0.5391,0.4756,0.3350,0.5337,0.5334,0.5336
6,LR,0.7260,0.0141,0.5026,0.3632,0.2733,0.5177,0.5127,0.5152
7,LR_balanced,0.7263,0.0139,0.5005,0.4799,0.5823,0.5458,0.4837,0.5129


### 2. 固定方案，再评估原测试集

选择 OOF F1 最高的方案，使用全部 `X_train` 重新拟合。保留从 OOF 选定的精确阈值，不根据测试集结果回调。
同时显示原始基线、随机森林的 OOF 阈值版本、选中模型的 0.5 阈值版本，以区分模型和阈值各自带来的变化。


In [12]:
selected_name = cv_comparison.loc[0, "model"]
selected_threshold = float(cv_comparison.loc[0, "threshold"])
selected_model = clone(candidates[selected_name])
with threadpool_limits(limits=4):
    selected_model.fit(X_train, y_train)

y_prob_selected = selected_model.predict_proba(X_test)[:, 1]
y_pred_selected = (y_prob_selected >= selected_threshold).astype(int)
print(f"Selected: {selected_name}; threshold={selected_threshold:.6f}")


Selected: HGB_leaf31; threshold=0.277095


In [13]:
def evaluation_row(name, truth, prediction, probability, threshold):
    tn, fp, fn, tp = confusion_matrix(truth, prediction, labels=[0, 1]).ravel()
    return {
        "model": name, "threshold": threshold,
        "accuracy": accuracy_score(truth, prediction),
        "precision_1": precision_score(truth, prediction, zero_division=0),
        "recall_1": recall_score(truth, prediction),
        "f1_1": f1_score(truth, prediction),
        "roc_auc": roc_auc_score(truth, probability),
        "average_precision": average_precision_score(truth, probability),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


rf_oof_threshold = float(cv_comparison.set_index("model").loc["RF", "threshold"])
test_comparison = pd.DataFrame([
    evaluation_row("LR", y_test, y_pred_lr, y_prob_lr, 0.5),
    evaluation_row("LR_balanced", y_test, y_pred_lr_balanced, y_prob_lr_balanced, 0.5),
    evaluation_row("RF", y_test, y_pred_rf, y_prob_rf, 0.5),
    evaluation_row("RF_OOF_threshold", y_test,
                   (y_prob_rf >= rf_oof_threshold).astype(int), y_prob_rf, rf_oof_threshold),
    evaluation_row(selected_name + "_default", y_test,
                   selected_model.predict(X_test), y_prob_selected, 0.5),
    evaluation_row(selected_name + "_OOF_threshold", y_test,
                   y_pred_selected, y_prob_selected, selected_threshold),
])
display(test_comparison.round(4))
print("Selected model confusion matrix:")
print(confusion_matrix(y_test, y_pred_selected, labels=[0, 1]))
print(classification_report(y_test, y_pred_selected, digits=4))


,model,threshold,accuracy,precision_1,recall_1,f1_1,roc_auc,average_precision,tn,fp,fn,tp
0,LR,0.5000,0.8077,0.6868,0.2396,0.3553,0.7076,0.4939,4528,145,1009,318
1,LR_balanced,0.5000,0.6797,0.3672,0.6202,0.4613,0.7081,0.4904,3255,1418,504,823
2,RF,0.5000,0.8123,0.6310,0.3647,0.4623,0.7552,0.5358,4390,283,843,484
3,RF_OOF_threshold,0.3350,0.7902,0.5261,0.5162,0.5211,0.7552,0.5358,4056,617,642,685
4,HGB_leaf31_default,0.5000,0.8182,0.6582,0.3700,0.4737,0.7801,0.5557,4418,255,836,491
5,HGB_leaf31_OOF_threshold,0.2771,0.7868,0.5164,0.5705,0.5421,0.7801,0.5557,3964,709,570,757


Selected model confusion matrix:
[[3964  709]
 [ 570  757]]
              precision    recall  f1-score   support

           0     0.8743    0.8483    0.8611      4673
           1     0.5164    0.5705    0.5421      1327

    accuracy                         0.7868      6000
   macro avg     0.6953    0.7094    0.7016      6000
weighted avg     0.7951    0.7868    0.7905      6000



### 3. 保存实验记录

`credit_default_cv.csv` 是训练集内部的开发结果，`credit_default_comparison.csv` 是固定方案在原测试集上的对照结果。
原测试集已经用于多次探索，不把这一轮分数描述为全新独立测试的无偏结论。

方法参考：[sklearn 分类阈值选择](https://scikit-learn.org/stable/modules/classification_threshold.html)、[梯度提升树](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting)。


In [14]:
results_dir = project_root / "Week02_ML_Practical_Projects" / "results"
results_dir.mkdir(exist_ok=True)
cv_comparison.to_csv(results_dir / "credit_default_cv.csv", index=False)
test_comparison.to_csv(results_dir / "credit_default_comparison.csv", index=False)
selection_record = {
    "selection_metric": "positive-class OOF F1",
    "cv": "3-fold stratified CV inside the original 80% training set; shuffle=True, random_state=42",
    "selected_model": selected_name,
    "selected_threshold": selected_threshold,
    "selected_estimator": repr(selected_model),
    "train_rows": len(y_train), "test_rows": len(y_test),
    "caveat": "OOF scores are used for selection; original test set was already inspected in earlier experiments.",
}
(results_dir / "credit_default_selection.json").write_text(
    json.dumps(selection_record, ensure_ascii=False, indent=2) + "\n",
)
print("Saved CV comparison, test comparison and selection record to:", results_dir)


Saved CV comparison, test comparison and selection record to: /Users/leohome/AI-Internship-Qianyu/Week02_ML_Practical_Projects/results


### 本轮结果小结（2026-11-01）

训练集内比较了 8 个候选方案，按 OOF 违约 F1 选择 **HGB_leaf31**，阈值为 **0.277095**。
HGB_leaf15 和 HGB_leaf31 的 OOF F1 非常接近，不能将两者的微小分差解读为稳定优势。

| 方案 | Accuracy | 违约 Precision | 违约 Recall | 违约 F1 | ROC-AUC |
| --- | ---: | ---: | ---: | ---: | ---: |
| LR | 80.77% | 68.68% | 23.96% | 35.53% | 0.7076 |
| RF | 81.23% | 63.10% | 36.47% | 46.23% | 0.7552 |
| HGB_leaf31_default | 81.82% | 65.82% | 37.00% | 47.37% | 0.7801 |
| HGB_leaf31_OOF_threshold | 78.68% | 51.64% | 57.05% | 54.21% | 0.7801 |

- 同一个模型降低阈值后，Recall 和 F1 提高，但误报增加、Accuracy 降低；根据目标选择取舍。
- 仅调整阈值不改变预测概率的排序，所以同一个模型这两行的 ROC-AUC 相同。
- 仅改变 LR 的预处理，交叉验证 AUC 也有改善；模型层数不是唯一的改进方向。
- 以上是当前数据划分的探索性结果，原测试集此前已查看多次；不据此宣称全新独立测试的泛化表现。
